## Imports

In [8]:
import pinocchio as pin ##type: ignore
from pinocchio.visualize import MeshcatVisualizer ##type: ignore
import numpy as np
from numpy.linalg import norm, solve
import sys
import time

from integrate import rk4_step, wrap_to_pi
from friction import tau_fric
from trajectories import circular_trajectory
from inverse_kinematics import solve_ik, solve_ik_trajectory
from spline import cubic_spline_interpolation
from control import pd_control, critically_damped_gains, tvlqr_backward_pass
from linearize import linearize_discrete, linearize_trajectory
from disturbances import random_torque

## loading model and date

In [9]:
model, collision_model, visual_model = pin.buildModelsFromUrdf(
    "lbr_iiwa7_r800.urdf", package_dirs="."
)
data = model.createData()

## Initializing visualizer (meshcat)

In [10]:
try:
    viz = MeshcatVisualizer(model, collision_model, visual_model)
    viz.initViewer(open=True)
    viz.loadViewerModel()
except ImportError as err:
    print(
        "Error while initializing the viewer. "
        "It seems you should install Python meshcat"
    )
    print(err)
    sys.exit(0)
    

You can open the visualizer by visiting the following URL:
http://127.0.0.1:7002/static/


## Circular task-space trajectory -> joint-space trajectory (IK)

In [11]:
T = 5
dt = 0.001
poses = circular_trajectory(radius=0.1, T=T, dt=dt)
JOINT_ID = 7
q = pin.randomConfiguration(model)
q_traj, successes = solve_ik_trajectory(model, data, JOINT_ID, poses, q)

print(f"{len(poses)} waypoints, {sum(successes)}/{len(successes)} converged")

# for q_i in q_traj:
#     viz.display(q_i)
#     time.sleep(0.05)

waypoint    0: iters= 169  |err|=9.38e-08  success=True
waypoint   10: iters=  73  |err|=9.07e-08  success=True
waypoint   20: iters=  72  |err|=9.95e-08  success=True
waypoint   30: iters=  72  |err|=9.90e-08  success=True
waypoint   40: iters=  72  |err|=9.86e-08  success=True
waypoint   50: iters=  72  |err|=9.83e-08  success=True
waypoint   60: iters=  72  |err|=9.81e-08  success=True
waypoint   70: iters=  72  |err|=9.80e-08  success=True
waypoint   80: iters=  72  |err|=9.80e-08  success=True
waypoint   90: iters=  72  |err|=9.81e-08  success=True
waypoint  100: iters=  72  |err|=9.83e-08  success=True
waypoint  110: iters=  72  |err|=9.86e-08  success=True
waypoint  120: iters=  72  |err|=9.90e-08  success=True
waypoint  130: iters=  72  |err|=9.95e-08  success=True
waypoint  140: iters=  73  |err|=9.07e-08  success=True
waypoint  150: iters=  73  |err|=9.13e-08  success=True
waypoint  160: iters=  73  |err|=9.20e-08  success=True
waypoint  170: iters=  73  |err|=9.27e-08  succe

## Use a spline to fit q, q_d, and q_dd

In [12]:
t = np.linspace(0, T, round(T / dt) + 1)

q, q_d, q_dd = cubic_spline_interpolation(t, q_traj)


## Sim
### Model has no fric or disturbance's
### Use pinnochio recurisve newton euler alg (pin.rnea) to get FF torques
### Use PD control as well (without it, numerical drift causes instability)

In [13]:
data_sim = model.createData()

q_sim = q[0].copy()
v_sim = q_d[0].copy()

Kp, Kd = critically_damped_gains(model, data_sim, q[0], wn=20.0, zeta=1.0)

for k in range(len(t)):
    tau_ff = pin.rnea(model, data_sim, q[k], q_d[k], q_dd[k])
    tau_dist = random_torque(model)
    q_sim, v_sim = rk4_step(q_sim, v_sim, tau_ff + pd_control(q_sim, q[k], v_sim, Kp, Kd) + tau_dist, model, data_sim, dt)
    if k % 2 == 0:
        viz.display(q_sim)
    # time.sleep(dt)

### Note on above: had to do tricky pd tuning to prevent instability in the same. 
### LQR does not have this issue.  

## TVLQR tracking sim
### Same FF (pin.rnea) as above, but feedback is -K[k] @ (x - x_ref) from
### the TVLQR backward pass instead of a hand-tuned PD gain

In [14]:
# re-initialize state
data_sim = model.createData()
q_sim = q[0].copy()
v_sim = q_d[0].copy()

# FF torque along the whole reference trajectory -> linearize about it -> TVLQR gains
tau_traj = np.array([pin.rnea(model, data_sim, q[k], q_d[k], q_dd[k]) for k in range(len(t))])
A_traj, B_traj = linearize_trajectory(model, data_sim, q, q_d, tau_traj, dt)

nv = model.nv
Q = np.eye(2 * nv)
Q[:nv, :nv] *= 100  # weight position error more than velocity error
R = np.eye(nv) * 0.01
Qf = Q

K_list = tvlqr_backward_pass(A_traj, B_traj, Q, R, Qf)

for k in range(len(t)):
    tau_ff = pin.rnea(model, data_sim, q[k], q_d[k], q_dd[k])
    x_sim = np.concatenate([q_sim, v_sim])
    x_ref = np.concatenate([q[k], q_d[k]])
    tau_fb = -K_list[k] @ (x_sim - x_ref)
    tau_dist = random_torque(model)
    q_sim, v_sim = rk4_step(q_sim, v_sim, tau_ff + tau_fb + tau_dist, model, data_sim, dt)
    if k % 2 == 0:
        viz.display(q_sim)
    # time.sleep(dt)